### CART from PCA v3 dataset
This CART model uses the same subject-safe split and GroupKFold validation as the newest XGBoost workflow.


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('../dataset/PCA_datasets/full_v3.csv')
df.head()


In [ ]:
X = df.drop(['Activity', 'subject'], axis=1)
y = df['Activity']
le = LabelEncoder()
y = le.fit_transform(y)
groups = df['subject']

print('X shape:', X.shape)
print('num classes:', len(le.classes_))
print('subjects in dataset:', groups.nunique())


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=69)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups.iloc[train_idx]

train_subjects = sorted(groups_train.unique())
test_subjects = sorted(groups.iloc[test_idx].unique())
print('Training subjects:', train_subjects)
print('Test subjects:', test_subjects)
print('Overlap:', set(train_subjects) & set(test_subjects))


In [ ]:
gkf = GroupKFold(n_splits=5)
cart = DecisionTreeClassifier(random_state=69)

cv_scores = cross_val_score(
    cart,
    X_train,
    y_train,
    cv=gkf,
    groups=groups_train,
    scoring='f1_macro'
)

print('CV F1 scores:', cv_scores)
print('Mean CV F1:', cv_scores.mean())


In [ ]:
cart.fit(X_train, y_train)
y_pred = cart.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
cm = confusion_matrix(y_test, y_pred, normalize='true')
labels = le.classes_

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='.1%', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label (Ground Truth)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
